In [3]:
A1 = [[1,2,3],
      [4,5,6]]

A2 = [[7,8,9],
      [1,1,1],
      [2,2,2]]

A3 = [[3,3,3]]

B = [[1,0],
     [0,1],
     [1,1]]
A1, A2, A3, B

([[1, 2, 3], [4, 5, 6]],
 [[7, 8, 9], [1, 1, 1], [2, 2, 2]],
 [[3, 3, 3]],
 [[1, 0], [0, 1], [1, 1]])

In [9]:
def pack_matrices(A_list):
    A_packed = []
    offsets = [0]

    for A in A_list:
        A_packed.extend(A)  # add rows
        offsets.append(len(A_packed))
    print("Packed A:", A_packed)

    return A_packed, offsets

In [10]:
def matmul(A, B):
    m, k = len(A), len(A[0])
    k2, n = len(B), len(B[0])

    C = [[0 for _ in range(n)] for _ in range(m)]

    for i in range(m):
        for j in range(n):
            for p in range(k):
                C[i][j] += A[i][p] * B[p][j]

    return C

In [11]:
def grouped_gemm_with_offsets(A_list, B):
    A_packed, offsets = pack_matrices(A_list)

    results = []

    for g in range(len(A_list)):
        start = offsets[g]
        end = offsets[g+1]

        A_chunk = A_packed[start:end]
        C_chunk = matmul(A_chunk, B)

        results.append(C_chunk)

    return results, offsets

In [12]:
C_list, offsets = grouped_gemm_with_offsets([A1, A2, A3], B)

print("Offsets:", offsets)
for i, C in enumerate(C_list):
    print(f"C{i+1} =", C)

Packed A: [[1, 2, 3], [4, 5, 6], [7, 8, 9], [1, 1, 1], [2, 2, 2], [3, 3, 3]]
Offsets: [0, 2, 5, 6]
C1 = [[4, 5], [10, 11]]
C2 = [[16, 17], [2, 2], [4, 4]]
C3 = [[6, 6]]


In [13]:
C_list

[[[4, 5], [10, 11]], [[16, 17], [2, 2], [4, 4]], [[6, 6]]]

In [7]:
matmul(A1, B)

[[4, 5], [10, 11]]

In [8]:
matmul(A2, B)

[[16, 17], [2, 2], [4, 4]]

In [ ]:
Z = [[1, 2, 3], [4, 5, 6], [7, 8, 9], [1, 1, 1], [2, 2, 2], [3, 3, 3]] ## this shows packaging of all 3 


In [15]:
matmul(Z, B)

[[4, 5], [10, 11], [16, 17], [2, 2], [4, 4], [6, 6]]

In [4]:
import numpy as np

K, N = 4, 5
Ms = [3, 4, 2]   # tokens routed to each expert

    # Expert weight matrices (fixed per expert)
B_list = [np.random.randn(K, N).astype(np.float32) for _ in range(3)]

    # Token activation matrices (different M per group)
A_list = [np.random.randn(m, K).astype(np.float32) for m in Ms]

B_list, A_list

([array([[-0.31940976,  0.5816137 ,  2.233721  ,  0.49819976,  0.31941935],
         [ 0.13639107,  0.42888743, -0.6517263 ,  0.25311464,  1.4292314 ],
         [ 0.27635145, -0.4565226 , -0.9029428 ,  0.35509902, -0.09999506],
         [ 0.3817162 , -0.47486869, -0.3769192 , -1.8162441 , -0.7019967 ]],
        dtype=float32),
  array([[ 0.94946045,  0.15580909, -0.2599191 , -1.7405691 ,  0.5659204 ],
         [-0.13015294,  0.11976762,  0.59114796,  2.1290534 , -1.6265802 ],
         [ 0.05997131, -0.46394446, -1.7731953 ,  0.2950701 , -0.56498563],
         [-0.17052476, -1.2123932 ,  0.60916936, -0.27633864,  0.00883607]],
        dtype=float32),
  array([[ 0.21307352,  0.7895878 , -0.44325334, -0.8813927 ,  0.78368706],
         [-0.13462918, -0.50211346, -0.20563962,  2.2378552 ,  0.18059435],
         [ 0.71316564, -0.695059  , -0.9892321 , -1.2217853 , -0.27605087],
         [ 0.22673698,  1.5158033 , -0.10510095,  1.3203617 , -1.1732458 ]],
        dtype=float32)],
 [array([[ 1

In [18]:
for i, (A, B) in enumerate(zip(A_list, B_list)):
    print(A.shape, B.shape)

(3, 4) (4, 5)
(4, 4) (4, 5)
(2, 4) (4, 5)


In [19]:
A_flat = np.concatenate(A_list, axis=0)
A_flat

array([[-1.5157347 ,  0.29540178,  0.34062833,  0.213833  ],
       [ 0.48954934,  0.45100322,  0.19693962,  0.53773123],
       [ 0.89566934, -0.5946875 , -0.05793198,  0.71528685],
       [ 0.61756337, -0.18707044, -0.36002958, -0.7591559 ],
       [ 0.82022625,  0.40027565, -0.24439456, -0.20827107],
       [ 0.7144279 , -0.49147567, -1.0307139 , -1.3499675 ],
       [ 1.5375658 , -1.7216918 , -0.42481524, -0.34511805],
       [ 1.0585083 ,  0.23933755,  0.5789165 , -0.53362465],
       [ 1.6032212 , -1.4560893 , -0.7972034 ,  0.5810468 ]],
      dtype=float32)

In [5]:
a_flat = np.concatenate([A1, A2, A3], axis=0)
a_flat

array([[1, 2, 3],
       [4, 5, 6],
       [7, 8, 9],
       [1, 1, 1],
       [2, 2, 2],
       [3, 3, 3]])

In [6]:
b = np.array(B)
b

array([[1, 0],
       [0, 1],
       [1, 1]])

In [26]:
ans = a_flat @ b
ans

array([[ 4,  5],
       [10, 11],
       [16, 17],
       [ 2,  2],
       [ 4,  4],
       [ 6,  6]])

In [35]:
anss = []
a_dims = np.array([0,len(A1), len(A2), len(A3)])
offsets = np.cumsum(a_dims)

a_dims, offsets

(array([0, 2, 3, 1]), array([0, 2, 5, 6]))

In [36]:
for i in range(len(offsets)-1):
    anss.append(ans[offsets[i]:offsets[i+1]])

In [37]:
anss

[array([[ 4,  5],
        [10, 11]]),
 array([[16, 17],
        [ 2,  2],
        [ 4,  4]]),
 array([[6, 6]])]

In [1]:
import torch 

In [9]:
mat_a = torch.tensor(a_flat, dtype=torch.float32)
mat_b = torch.tensor(b, dtype=torch.float32)
mat_a, mat_b

(tensor([[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.],
         [1., 1., 1.],
         [2., 2., 2.],
         [3., 3., 3.]]),
 tensor([[1., 0.],
         [0., 1.],
         [1., 1.]]))

In [21]:
G = 3          # number of groups / experts
K = 8          # input  dim (shared across all groups)
N = 6          # output dim (shared across all groups)
Ms = [3, 5, 2] # tokens routed to expert 0, 1, 2
total_M = sum(Ms)  # = 10

print("─" * 52)
print("  F.grouped_mm  —  shape rules explained")
print("─" * 52)

# ══════════════════════════════════════════════════════
# MODE A: 3D mat_a, 3D mat_b, offs=None
#   mat_a : (G, M, K)   — same M per group (padded if needed)
#   mat_b : (G, N, K)   — note: N before K  (transposed weights)
#   output: (G, M, N)
# ══════════════════════════════════════════════════════
print("\n── MODE A: 3D × 3D, offs=None ──────────────────")
print("  mat_a: (G, M, K)  →  (3, 4, 8)  [pad shorter groups]")
print("  mat_b: (G, N, K)  →  (3, 6, 8)  ← N before K!")
print("  offs : None")
print("  out  : (G, M, N)  →  (3, 4, 6)")

M_padded = 4   # pad all groups to the same M for 3D mode
mat_a_3d = torch.randn(G, M_padded, K, dtype=torch.float32)
mat_b_3d = torch.randn(G, N, K, dtype=torch.float32)

print("mat_a_3d:", mat_a_3d)
print("mat_b_3d:", mat_b_3d)

────────────────────────────────────────────────────
  F.grouped_mm  —  shape rules explained
────────────────────────────────────────────────────

── MODE A: 3D × 3D, offs=None ──────────────────
  mat_a: (G, M, K)  →  (3, 4, 8)  [pad shorter groups]
  mat_b: (G, N, K)  →  (3, 6, 8)  ← N before K!
  offs : None
  out  : (G, M, N)  →  (3, 4, 6)
mat_a_3d: tensor([[[ 0.0099,  0.0106, -0.7005, -2.4683,  0.4896, -1.1943,  1.1056,
          -0.2108],
         [ 2.2866,  0.0614,  0.4960, -0.2261,  0.5719,  0.3492,  0.2701,
           1.4317],
         [ 0.4748, -0.9592,  0.6185, -0.6376, -0.2012,  0.5819,  0.0129,
          -0.4387],
         [ 1.1466,  1.3019, -0.3947, -0.1298, -1.5017,  1.5889, -0.2993,
          -0.5784]],

        [[ 1.0365, -0.0283, -1.0819,  0.4961, -0.0348,  0.4931, -1.4437,
          -0.8563],
         [-0.2008,  0.8879,  1.1887,  0.0412, -1.7348, -0.9726,  0.1342,
          -0.7510],
         [ 1.1420, -0.3882,  0.5361,  0.7268,  0.0421, -1.0292,  0.0658,
          

In [22]:
torch.nn.functional.grouped_mm(mat_a_3d, mat_b_3d,offs=None)

RuntimeError: contraction dimension of mat_a and mat_b must match

## Grouped GEMM: clear input/output examples

Below are two working examples that print exact inputs and outputs:

1. **Uniform groups (3D mode, `offs=None`)**
2. **Ragged groups (packed mode, `offs` provided)`

These are the practical patterns used in MoE kernels.

In [29]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)

print("=== Example 1: 3D grouped_mm (offs=None) ===")
# Working shape combo in this runtime:
#   mat_a: (G, M, K)
#   mat_b: (G, K, N)
#   out  : (G, M, N)
# Keep K,N multiples of 4 for alignment.

G, M, K, N = 3, 2, 4, 4

A_3d = torch.tensor(
    [
        [[1., 2., 3., 4.], [4., 5., 6., 7.]],
        [[1., 0., 1., 0.], [0., 1., 1., 1.]],
        [[2., 2., 0., 1.], [1., 3., 1., 0.]],
    ],
    dtype=torch.float32,
).contiguous()

B_3d = torch.tensor(
    [
        [[1., 0., 1., 0.], [0., 1., 1., 0.], [1., 1., 0., 1.], [0., 0., 1., 1.]],
        [[2., 1., 0., 1.], [1., 0., 1., 0.], [0., 1., 1., 1.], [1., 1., 0., 0.]],
        [[1., 2., 0., 1.], [2., 1., 1., 0.], [0., 1., 1., 1.], [1., 0., 2., 1.]],
    ],
    dtype=torch.float32,
).contiguous()  # (3, 4, 4)

C_3d = torch.nn.functional.grouped_mm(A_3d, B_3d, offs=None)
C_3d_ref = torch.stack([A_3d[g] @ B_3d[g] for g in range(G)], dim=0)

print("A_3d shape:", tuple(A_3d.shape))
print("B_3d shape:", tuple(B_3d.shape))
print("C_3d shape:", tuple(C_3d.shape))
print("C_3d:\n", C_3d)
print("matches per-group reference:", torch.allclose(C_3d, C_3d_ref))

print("\n=== Example 2: packed grouped_mm with offs (ragged M per group) ===")
# Group rows: M0=2, M1=3, M2=1
A0 = torch.tensor([[1., 2., 3., 4.], [4., 5., 6., 7.]], dtype=torch.float32)
A1 = torch.tensor([[1., 0., 1., 0.], [0., 1., 1., 1.], [2., 1., 0., 1.]], dtype=torch.float32)
A2 = torch.tensor([[2., 2., 0., 1.]], dtype=torch.float32)
A_pad = torch.tensor([[0., 0., 0., 0.]], dtype=torch.float32)  # ignored tail
A_flat = torch.cat([A0, A1, A2, A_pad], dim=0).contiguous()    # (7, 4)

B_group = B_3d.clone().contiguous()  # (3, 4, 4)
# offs[i] = end of group i. len(offs)=num_groups and offs[-1] < total rows
offs = torch.tensor([2, 5, 6], dtype=torch.int32)

C_flat = torch.nn.functional.grouped_mm(A_flat, B_group, offs=offs)
C_valid = C_flat[: int(offs[-1])]  # valid output rows only

print("A_flat shape:", tuple(A_flat.shape), "(last row ignored)")
print("B_group shape:", tuple(B_group.shape))
print("offs (int32):", offs.tolist())
print("C_flat shape:", tuple(C_flat.shape))
print("C_valid (rows [0:offs[-1]]):\n", C_valid)

# split output back per group and verify
starts = [0, int(offs[0]), int(offs[1])]
ends = [int(offs[0]), int(offs[1]), int(offs[2])]
chunks = [C_flat[s:e] for s, e in zip(starts, ends)]
refs = [A0 @ B_group[0], A1 @ B_group[1], A2 @ B_group[2]]

for i, (c, r) in enumerate(zip(chunks, refs)):
    print(f"group {i}: out shape={tuple(c.shape)}, matches={torch.allclose(c, r)}")
    print(c)


=== Example 1: 3D grouped_mm (offs=None) ===
A_3d shape: (3, 2, 4)
B_3d shape: (3, 4, 4)
C_3d shape: (3, 2, 4)
C_3d:
 tensor([[[ 4.,  5.,  7.,  7.],
         [10., 11., 16., 13.]],

        [[ 2.,  2.,  1.,  2.],
         [ 2.,  2.,  2.,  1.]],

        [[ 7.,  6.,  4.,  3.],
         [ 7.,  6.,  4.,  2.]]])
matches per-group reference: True

=== Example 2: packed grouped_mm with offs (ragged M per group) ===
A_flat shape: (7, 4) (last row ignored)
B_group shape: (3, 4, 4)
offs (int32): [2, 5, 6]
C_flat shape: (7, 4)
C_valid (rows [0:offs[-1]]):
 tensor([[ 4.,  5.,  7.,  7.],
        [10., 11., 16., 13.],
        [ 2.,  2.,  1.,  2.],
        [ 2.,  2.,  2.,  1.],
        [ 6.,  3.,  1.,  2.],
        [ 7.,  6.,  4.,  3.]])
group 0: out shape=(2, 4), matches=True
tensor([[ 4.,  5.,  7.,  7.],
        [10., 11., 16., 13.]])
group 1: out shape=(3, 4), matches=True
tensor([[2., 2., 1., 2.],
        [2., 2., 2., 1.],
        [6., 3., 1., 2.]])
group 2: out shape=(1, 4), matches=True
tensor

In [26]:
import inspect
print(torch.nn.functional.grouped_mm.__doc__)


    grouped_mm(mat_a, mat_b, *, offs=None, bias=None, out_dtype=None)

    Computes a grouped matrix multiply that shares weight shapes across experts but
    allows jagged token counts per expert, which is common in Mixture-of-Experts
    (MoE) layers. Both ``mat_a`` and ``mat_b`` must be 2D or 3D tensors that already
    satisfy the physical layout restrictions of grouped GEMM kernels (e.g., row-major
    ``mat_a`` and column-major ``mat_b`` for FP8 inputs). Inputs are currently
    expected to be ``torch.bfloat16`` values on CUDA devices with :math:`SM \ge 80`.

    Args:
        mat_a: Left operand. When 2D, its leading dimension is sliced into groups
            according to ``offs``. When 3D, its first dimension enumerates the groups
            directly and ``offs`` must be ``None``.
        mat_b: Right operand. When both operands are 2D (e.g., MoE weight-gradient
            updates), the trailing dimension of ``mat_a`` and the leading dimension of
            ``mat_b`` are p